# Data Audit and Exploratory Data Analysis

This notebook loads the merged Bluesky/Yahoo Finance dataset, audits data quality, and builds the main EDA visuals for the final project.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_merged_data, summarize_data_quality
from src.plots import set_plot_style

set_plot_style()

df = load_merged_data()
display(df.head())


## Dataset Shape and Quality Checks

These checks verify row count, nulls, ticker coverage, duplicate keys, and timestamp coverage before modeling.

In [ ]:
quality = summarize_data_quality(df)
print(quality)

print("\n--- Summary Statistics ---")
display(df.describe())

print("\n--- Null Value Counts ---")
print(df.null_count())

print("\n--- Ticker Value Counts ---")
print(df["Ticker"].value_counts().sort("count", descending=True))


## Timestamp Coverage

This project combines 24/7 social posts with market-hour financial data. We need to make sure old social posts were not accidentally paired with recent backfilled market values.

In [ ]:
year_counts = (
    df.with_columns(pl.col("Timestamp").dt.year().alias("Year"))
    .group_by("Year")
    .len()
    .sort("Year")
)
display(year_counts)

per_ticker_range = (
    df.group_by("Ticker")
    .agg(
        [
            pl.col("Timestamp").min().alias("min_ts"),
            pl.col("Timestamp").max().alias("max_ts"),
            pl.len().alias("rows"),
        ]
    )
    .sort("min_ts")
)
display(per_ticker_range)


## Distribution Visuals

In [ ]:
from src.plots import plot_sentiment_and_volume_distributions

pdf = df.to_pandas()
fig = plot_sentiment_and_volume_distributions(pdf)
plt.show()


In [ ]:
from src.plots import plot_sentiment_volume_scatter

fig = plot_sentiment_volume_scatter(pdf)
plt.show()


## Time Series Example

In [ ]:
ticker_df = df.filter(pl.col("Ticker") == "$AAPL").sort("Timestamp")

if not ticker_df.is_empty():
    ticker_df = ticker_df.with_columns(
        pl.col("Sentiment").rolling_mean(window_size=5).alias("Sentiment_MA_5")
    )
    ticker_pdf = ticker_df.to_pandas()

    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax1.set_xlabel("Timestamp")
    ax1.set_ylabel("Close Price", color="tab:blue")
    ax1.plot(ticker_pdf["Timestamp"], ticker_pdf["Close"], color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.set_ylabel("Post Sentiment (MA 5)", color="tab:red")
    ax2.plot(ticker_pdf["Timestamp"], ticker_pdf["Sentiment_MA_5"], color="tab:red", alpha=0.6)
    ax2.tick_params(axis="y", labelcolor="tab:red")

    fig.tight_layout()
    plt.title("AAPL: Hourly Closing Price vs Smoothed Sentiment")
    plt.show()


## EDA Findings

- Sentiment is heavily concentrated around neutral values, so raw sentiment alone is likely weak.
- Trading volume is right-skewed, so scaling or anomaly features should help modeling.
- The dataset has enough rows after cleaning, but the old timestamp range needs filtering before final modeling.
- The current post-level rows should be aggregated to ticker-hour rows before serious modeling, otherwise hours with many posts are over-weighted.